# AksharaPiece on Kaggle: week 1 experiments

This notebook runs the first experiments of the **AksharaPiece** project (syllable-aware tokenization for Nepali):

1. Check the akshara splitter (38 tests)
2. Audit existing tokenizers on FLORES Nepali–English, including Hugging Face models such as MuRIL, IndicBERT v2 and Sarvam-1
3. Show where each tokenizer cuts through a syllable
4. Download and clean Nepali text from IndicCorp v2
5. Audit all tokenizers on held-out IndicCorp text
6. Train AksharaPiece and plain BPE/Unigram tokenizers on the same 100 MB and compare them
7. Save every result for download

**Before you press Run All**

1. **Settings → Internet → On.** Kaggle asks you to verify your phone number the first time.
2. **Accelerator: None.** Everything here runs on CPU.
3. *Optional:* to include gated tokenizers (Gemma, Llama 3), add a Hugging Face token: **Add-ons → Secrets → Add secret** named `HF_TOKEN`, and accept each model's licence on huggingface.co. Without it, those tokenizers are skipped and everything else still runs.

Expected run time: about 45–75 minutes with the default settings below. Lower `RAW_MB` and `TOKENIZER_MB` for a faster first run.

In [ ]:
# ---- Settings: change these if you want a faster or bigger run ----
RAW_MB = 400            # Nepali text to download from IndicCorp v2 (MB)
HELDOUT_MB = 20         # held-out text, never used for training (MB)
TOKENIZER_MB = 100      # text used to train the pilot tokenizers (MB)
AUDIT_SENTENCES = 10000 # held-out sentences used to score each tokenizer
PILOT_VOCABS = [8000, 16000, 32000]
PILOT_MODELS = ["bpe", "unigram"]
HF_TOKENIZERS = [
    "google/muril-base-cased",
    "ai4bharat/IndicBERTv2-MLM-only",
    "sarvamai/sarvam-1",
    "Qwen/Qwen2.5-0.5B",
    "google/gemma-3-1b-it",       # gated: needs HF_TOKEN and an accepted licence
    "meta-llama/Llama-3.2-1B",    # gated
    # Add IndicSuperTokenizer here once you find its Hugging Face name in the paper.
]

## Setup
Unpacks the project code (embedded in this notebook), installs libraries and logs in to Hugging Face if you added a token.

In [ ]:
import base64, io, os, subprocess, sys, zipfile

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
os.chdir(WORK)
CODE_ZIP_B64 = "..."
if not os.path.exists(os.path.join(WORK, "aksharapiece", "scripts", "audit.py")):
    zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_ZIP_B64))).extractall(WORK)
PROJECT = os.path.join(WORK, "aksharapiece")
os.chdir(PROJECT)
for p in (PROJECT, os.path.join(PROJECT, "scripts")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTHONPATH"] = PROJECT


def run(cmd):
    """Run a shell command and stream its output into the notebook."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed: {cmd}")


print("Project code is in", PROJECT)

In [ ]:
%pip install -q sentencepiece tiktoken nepalitokenizers "mistral-common>=1.4" regex pytest

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("Logged in to Hugging Face.")
except Exception:
    print("No HF_TOKEN secret: gated tokenizers will be skipped (everything else still runs).")

## Step 1. Check the akshara splitter
The tests cover conjuncts, reph, nukta, ZWJ/ZWNJ, visarga, chandrabindu and word-final halant.

In [ ]:
run("python -m pytest -q tests")

from aksharapiece.akshara import aksharas, normalize
for w in ["नमस्ते", "क्षेत्र", "विद्यालयहरूमा", "काठमाडौं", "पढ्छन्", "स्त्री"]:
    print(f"{w:14s} ->", " | ".join(aksharas(normalize(w))))

## Step 2. Audit existing tokenizers on FLORES (Nepali–English)
6,511 unique Nepali sentences with English translations. **Akshara breaks** = share of token boundaries inside a syllable; **Words broken** = share of Nepali words with at least one such cut; **Parity vs EN** = Nepali tokens ÷ English tokens for the same sentences (not meaningful for Nepali-only tokenizers).

In [ ]:
run("python scripts/download_data.py --no-indiccorp")

D = "data/raw/wikipedia_en_ne_si_test_sets"
NE = " ".join(f"{D}/wikipedia.{s}.ne-en.ne" for s in ("dev", "devtest", "test"))
EN = " ".join(f"{D}/wikipedia.{s}.ne-en.en" for s in ("dev", "devtest", "test"))
HF_ARGS = " ".join(HF_TOKENIZERS)
run(f"python scripts/audit.py --ne {NE} --en {EN} --hf {HF_ARGS} --out results/audit_flores_v1")

In [ ]:
import pandas as pd


def show(csv_path, sort="fertility"):
    df = pd.read_csv(csv_path).sort_values(sort)
    out = pd.DataFrame({
        "Tokenizer": df["tokenizer"],
        "Vocab": df["vocab"].fillna("").map(lambda v: f"{int(v):,}" if v != "" else ""),
        "Tokens/word": df["fertility"].round(2),
        "Chars/token": df["chars_per_token"].round(2),
        "Akshara breaks": (100 * df["akshara_break_rate"]).round(1).astype(str) + "%",
        "Words broken": (100 * df["words_broken"]).round(1).astype(str) + "%",
        "UNK": (100 * df["unk_rate"]).round(2).astype(str) + "%",
        "Rényi eff.": df["renyi_eff"].round(3),
        "Parity vs EN": df["parity_vs_en"].round(2),
    })
    return out.reset_index(drop=True)


show("results/audit_flores_v1.csv")

## Step 3. See where each tokenizer cuts through a syllable
Red tokens start or end inside an akshara.

In [ ]:
import html
from IPython.display import HTML, display
from audit import load_default_adapters
from aksharapiece.akshara import akshara_boundaries

SENTENCE = normalize("विद्यालयहरूमा विद्यार्थीहरूले नेपाली भाषा पढ्छन् र काठमाडौंको इतिहास सिक्छन्।")
good = akshara_boundaries(SENTENCE, unit="byte")
raw = SENTENCE.encode("utf-8")
rows = []
for a in load_default_adapters(HF_TOKENIZERS):
    spans, _, _ = a.spans(SENTENCE)
    chips = []
    for s, e in spans:
        t = raw[s:e].decode("utf-8", "replace").replace(" ", "␣")
        bad = s not in good or e not in good
        style = "background:#fde2e1;color:#9b1c1c" if bad else "background:#e7f0ff;color:#1e3a8a"
        chips.append(f"<span style='{style};padding:1px 4px;margin:1px;border-radius:4px;display:inline-block'>{html.escape(t)}</span>")
    rows.append(f"<tr><td style='white-space:nowrap;padding-right:12px'><b>{html.escape(a.name)}</b><br>{len(spans)} tokens</td><td>{''.join(chips)}</td></tr>")
display(HTML("<table style='font-size:15px'>" + "".join(rows) + "</table>"))

## Step 4. Download and clean Nepali text (IndicCorp v2)
Streams the first `RAW_MB` megabytes of the Nepali file from Hugging Face (IndicCorp v2 is released under CC-0), then cleans it: normalization, filtering, deduplication, and removal of any FLORES sentence so test text never leaks into training.

In [ ]:
RAW_FILE = "data/raw/indiccorp_ne_sample.txt"
os.makedirs("data/raw", exist_ok=True)

if os.path.exists(RAW_FILE) and os.path.getsize(RAW_FILE) > 0:
    print(f"Using existing {RAW_FILE} ({os.path.getsize(RAW_FILE) / 1e6:.0f} MB)")
else:
    try:
        from huggingface_hub import HfFileSystem
        fs = HfFileSystem()
        repo = "datasets/ai4bharat/IndicCorpV2"
        candidates = [p for p in fs.ls(f"{repo}/data", detail=False) if p.rsplit("/", 1)[-1].startswith("ne")]
        src = candidates[0]
        print("streaming", src)
        limit, size = RAW_MB * 1_000_000, 0
        with fs.open(src, "rb", block_size=8 * 1024 * 1024) as f, open(RAW_FILE, "wb") as out:
            for line in f:
                out.write(line)
                size += len(line)
                if size >= limit:
                    break
        print(f"saved {size / 1e6:.0f} MB to {RAW_FILE}")
    except Exception as e:
        print("Could not download IndicCorp v2:", type(e).__name__, str(e)[:300])
        print("Check that Internet is On. If the dataset asks you to accept terms, accept them on "
              "huggingface.co/datasets/ai4bharat/IndicCorpV2 and add an HF_TOKEN secret.")

HAVE_TEXT = os.path.exists(RAW_FILE) and os.path.getsize(RAW_FILE) > 0

In [ ]:
import json

if HAVE_TEXT:
    run(f"python scripts/clean.py --inputs {RAW_FILE} --exclude '{D}/*.ne' --out data/clean "
        f"--heldout-mb {HELDOUT_MB} --tok-mb {TOKENIZER_MB} --pretrain-mb 100000")
    stats = json.load(open("data/clean/stats.json"))
    print(f"\nKept {stats['kept']:,} of {stats['read']:,} lines "
          f"({stats['duplicate']:,} duplicates, {stats['filtered']:,} filtered, {stats['test_overlap']:,} FLORES overlaps removed)")
else:
    print("Skipped: no IndicCorp text (see the message above).")

## Step 5. Audit every tokenizer on held-out IndicCorp text
News and web text, closer to everyday Nepali than Wikipedia. No English here, so parity is blank.

In [ ]:
if HAVE_TEXT:
    run(f"python scripts/audit.py --ne data/clean/heldout.txt --max-sentences {AUDIT_SENTENCES} "
        f"--hf {HF_ARGS} --out results/audit_indiccorp_heldout")
    display(show("results/audit_indiccorp_heldout.csv"))
else:
    print("Skipped: no IndicCorp text.")

## Step 6. AksharaPiece vs plain subwords, trained on the same text
For each vocabulary size, train a plain BPE/Unigram tokenizer (variant A) and an AksharaPiece tokenizer (variant B) on the same `TOKENIZER_MB` of text, then score both on held-out text. The question: how many extra tokens does keeping every syllable whole cost?

In [ ]:
if HAVE_TEXT:
    vocab_args = " ".join(map(str, PILOT_VOCABS))
    model_args = " ".join(PILOT_MODELS)
    run(f"python scripts/pilot.py --train-file data/clean/tokenizer_train.txt --train-mb {TOKENIZER_MB} "
        f"--test-file data/clean/heldout.txt --max-test {AUDIT_SENTENCES} "
        f"--vocab {vocab_args} --models {model_args} --out results/pilot_indiccorp")
    display(show("results/pilot_indiccorp.csv", sort="tokenizer"))
else:
    print("Skipped: no IndicCorp text.")

In [ ]:
import matplotlib.pyplot as plt

if HAVE_TEXT:
    df = pd.read_csv("results/pilot_indiccorp.csv")
    df["variant"] = df["tokenizer"].str.extract(r"^(A|B)\.")
    df["model"] = df["tokenizer"].str.extract(r"(BPE|UNIGRAM)")
    df["vocab"] = df["vocab"].astype(int)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for (variant, model), g in df.groupby(["variant", "model"]):
        g = g.sort_values("vocab")
        label = f"{'AksharaPiece' if variant == 'B' else 'Plain'} {model}"
        style = "-" if variant == "B" else "--"
        axes[0].plot(g["vocab"], g["fertility"], style, marker="o", label=label)
        axes[1].plot(g["vocab"], 100 * g["words_broken"], style, marker="o", label=label)
    axes[0].set(title="Tokens per word (lower is cheaper)", xlabel="Vocabulary size", xscale="log")
    axes[1].set(title="Words with a broken syllable (%)", xlabel="Vocabulary size", xscale="log")
    for ax in axes:
        ax.set_xticks(PILOT_VOCABS, [f"{v // 1000}k" for v in PILOT_VOCABS])
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.savefig("results/pilot_indiccorp.png", dpi=150)
    plt.show()

    for model in df["model"].unique():
        for v in sorted(df["vocab"].unique()):
            a = df[(df.variant == "A") & (df.model == model) & (df.vocab == v)].iloc[0]
            b = df[(df.variant == "B") & (df.model == model) & (df.vocab == v)].iloc[0]
            print(f"{model:8s} {v // 1000:>3}k: AksharaPiece uses {100 * (b.fertility / a.fertility - 1):+.1f}% tokens, "
                  f"broken words {100 * a.words_broken:.1f}% -> {100 * b.words_broken:.1f}%")

Also score the pilot tokenizers on FLORES, so they sit in the same table as GPT, Llama, MuRIL and the rest, with English parity.

In [ ]:
if HAVE_TEXT:
    run(f"python scripts/audit.py --ne {NE} --en {EN} --hf {HF_ARGS} "
        f"--tokenizer-dir results/pilot_indiccorp_tokenizers --out results/audit_flores_v1_with_pilot")
    display(show("results/audit_flores_v1_with_pilot.csv"))

## Step 7. Save the results
Creates `aksharapiece_results.zip` in the **Output** panel (right side) with every table, the chart and the trained tokenizers. To keep a permanent copy, click **Save Version → Save & Run All (Commit)**.

In [ ]:
import shutil

shutil.make_archive(os.path.join(WORK, "aksharapiece_results"), "zip", PROJECT, "results")
print("Saved", os.path.join(WORK, "aksharapiece_results.zip"))
for f in sorted(os.listdir("results")):
    print("  results/" + f)

## What to look at

- **Step 2 and Step 5 tables:** which tokenizers are cheapest for Nepali (tokens per word, parity) and which keep syllables whole (akshara breaks, words broken). Compare MuRIL, IndicBERT v2 and Sarvam-1 with the GPT and Llama tokenizers.
- **Step 6 chart:** the token cost of AksharaPiece at each vocabulary size. If the dashed and solid lines in the left chart meet at 16k–32k while the right chart stays near zero for AksharaPiece, syllable integrity is almost free.
- Send the `.md` files from `results/` back to continue with the next step: training small Transformer and Mamba models.